<a href="https://colab.research.google.com/github/bhaviii123/Air_Passenger_Forecasting_ML_vs_DL.ipynb/blob/main/my_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.The Multi-Scale Feature Extraction (MSFE).

This block "looks" at the image through two different lenses at once.

In [1]:
import torch
import torch.nn as nn

class MSFE(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MSFE, self).__init__()
        # Path 1: Fine details
        self.path1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        # Path 2: Coarse details
        self.path2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 2, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )
        # Fuse the paths back together
        self.fuse = nn.Conv2d(out_channels, out_channels, kernel_size=1)

    def forward(self, x):
        p1 = self.path1(x)
        p2 = self.path2(x)
        out = torch.cat([p1, p2], dim=1) # Concatenation
        return self.fuse(out)

2. The Residual Dense Block (RDB)

This is the "brain" of the model. It reuses features to ensure nothing is forgotten

In [2]:
class RDB(nn.Module):
    def __init__(self, channels, growth_rate=32):
        super(RDB, self).__init__()
        # Dense layers: Each layer sees the output of all previous layers
        self.conv1 = nn.Sequential(nn.Conv2d(channels, growth_rate, 3, 1, 1), nn.ReLU())
        self.conv2 = nn.Sequential(nn.Conv2d(channels + growth_rate, growth_rate, 3, 1, 1), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Conv2d(channels + 2*growth_rate, channels, 3, 1, 1))

    def forward(self, x):
        c1 = self.conv1(x)
        c2 = self.conv2(torch.cat([x, c1], 1)) # Concatenate input + c1
        c3 = self.conv3(torch.cat([x, c1, c2], 1)) # Concatenate all
        return c3 + x # Local Residual Learning (The "+" sign)

3. The Enhanced Attention Module (EAM)

This uses Max/Avg Pooling and Sigmoid to focus on important pixels (like edges)

In [5]:
class EAM(nn.Module):
    def __init__(self):
        super(EAM, self).__init__()
        # Look at the spatial relationship
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Max and Avg Pooling across the channel dimension
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        # 2. Concatenate and pass through Conv
        pool_feat = torch.cat([avg_out, max_out], dim=1)
        attention_map = self.sigmoid(self.conv(pool_feat))

        # 3. Multiply original features by the attention map
        return x * attention_map

4. The Full Architecture (Putting it Together)

In [4]:
class InternModel(nn.Module):
    def __init__(self):
        super(InternModel, self).__init__()
        self.msfe = MSFE(3, 64)

        # The Backbone
        self.rdb1 = RDB(64)
        self.rdb2 = RDB(64)
        self.rdb3 = RDB(64)

        # The Attention Filter
        self.eam = EAM()

        # Final reconstruction
        self.final_conv = nn.Conv2d(64, 3, kernel_size=3, padding=1)

    def forward(self, x):
        # 1. Start with MSFE
        feat_initial = self.msfe(x)

        # 2. Pass through RDBs with a "CNN Relay" (Skip Connection)
        r1 = self.rdb1(feat_initial)
        r2 = self.rdb2(r1)
        r3 = self.rdb3(r2)

        # Relay: Adding early features (r1) to deep features (r3)
        combined = r3 + r1

        # 3. Refine with Attention
        refined = self.eam(combined)

        # 4. Final Image Output
        return self.final_conv(refined)

2. Updated EAM (Spatial + Channel Attention)

In [4]:
import torch
import torch.nn as nn

class EnhancedEAM(nn.Module):
    def __init__(self, channels):
        super(EnhancedEAM, self).__init__()
        # Channel Attention
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.ca = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 1, padding=0),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, 1, padding=0),
            nn.Sigmoid()
        )
        # Spatial Attention (The one we built before)
        self.sa_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Apply Channel Attention
        y_ca = self.gap(x)
        y_ca = self.ca(y_ca)
        x = x * y_ca # Scale channels

        # 2. Apply Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y_sa = torch.cat([avg_out, max_out], dim=1)
        y_sa = self.sigmoid(self.sa_conv(y_sa))

        return x * y_sa # Scale pixels

3. Global Residual Learning (The "Identity" Path)

In [6]:
def forward(self, x):
        # Long Skip Connection (Save the original input)
        identity = x

        # 1. Feature Extraction
        feat = self.msfe(x)

        # 2. Deep Backbone (RDBs)
        r1 = self.rdb1(feat)
        r2 = self.rdb2(r1)
        r3 = self.rdb3(r2)

        # 3. Attention & Refinement
        refined = self.eam(r3 + r1)

        # 4. Reconstruction
        out = self.final_conv(refined)

        # 5. Global Residual: Final Image = Original + Learned Details
        return out + identity

1. The Complete Model Script (PyTorch)

In [7]:
import torch
import torch.nn as nn

# --- BLOCK 1: Multi-Scale Feature Extraction ---
class MSFE(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MSFE, self).__init__()
        self.path3x3 = nn.Conv2d(in_channels, out_channels // 2, 3, padding=1)
        self.path5x5 = nn.Conv2d(in_channels, out_channels // 2, 5, padding=2)
        self.relu = nn.ReLU(inplace=True)
        self.fuse = nn.Conv2d(out_channels, out_channels, 1)

    def forward(self, x):
        p1 = self.relu(self.path3x3(x))
        p2 = self.relu(self.path5x5(x))
        return self.fuse(torch.cat([p1, p2], dim=1))

# --- BLOCK 2: Residual Dense Block (The Engine) ---
class RDB(nn.Module):
    def __init__(self, channels, growth_rate=32):
        super(RDB, self).__init__()
        self.c1 = nn.Sequential(nn.Conv2d(channels, growth_rate, 3, 1, 1), nn.ReLU())
        self.c2 = nn.Sequential(nn.Conv2d(channels + growth_rate, growth_rate, 3, 1, 1), nn.ReLU())
        self.c3 = nn.Sequential(nn.Conv2d(channels + 2*growth_rate, channels, 3, 1, 1))

    def forward(self, x):
        out1 = self.c1(x)
        out2 = self.c2(torch.cat([x, out1], 1))
        out3 = self.c3(torch.cat([x, out1, out2], 1))
        return out3 + x  # Local Residual Learning

# --- BLOCK 3: Enhanced Attention Module (Spatial + Channel) ---
class EAM(nn.Module):
    def __init__(self, channels):
        super(EAM, self).__init__()
        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.ca = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, 1),
            nn.Sigmoid()
        )
        # Spatial Attention
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Apply Channel Attention
        x = x * self.ca(self.avg_pool(x))
        # Apply Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial = self.sa(torch.cat([avg_out, max_out], dim=1))
        return x * spatial

# --- THE FINAL ARCHITECTURE ---
class InternshipMasterModel(nn.Module):
    def __init__(self):
        super(InternshipMasterModel, self).__init__()
        self.msfe = MSFE(3, 64)

        # Backbone RDBs
        self.rdb1 = RDB(64)
        self.rdb2 = RDB(64)
        self.rdb3 = RDB(64)

        # Bridge (CNN Relay)
        self.relay = nn.Conv2d(64, 64, 1)

        # Attention
        self.eam = EAM(64)

        # Reconstruction
        self.final_conv = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x):
        initial_identity = x # Global Identity for Residual Learning

        feat = self.msfe(x)

        r1 = self.rdb1(feat)
        r2 = self.rdb2(r1)
        r3 = self.rdb3(r2)

        # CNN Relay: Jumping from RDB1 to RDB3
        out_backbone = self.relay(r1 + r3)

        # Attention Filter
        out_refined = self.eam(out_backbone)

        # Output + Global Residual
        return self.final_conv(out_refined) + initial_identity

3. How to Train This (The Code Snip)

In [8]:
model = InternshipMasterModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

# Example Training Loop Step
# optimizer.zero_grad()
# output = model(input_image)
# loss = criterion(output, ground_truth)
# loss.backward()
# optimizer.step()

In [9]:
import torch
import torch.nn as nn

# --- 1. Multi-Scale Feature Extraction (MSFE) ---
class MSFE(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        # Parallel paths for different detail levels
        self.path1 = nn.Conv2d(in_c, out_c // 2, kernel_size=3, padding=1)
        self.path2 = nn.Conv2d(in_c, out_c // 2, kernel_size=5, padding=2)
        self.relu = nn.ReLU(inplace=True)
        self.fuse = nn.Conv2d(out_c, out_c, kernel_size=1)

    def forward(self, x):
        p1 = self.relu(self.path1(x))
        p2 = self.relu(self.path2(x))
        return self.fuse(torch.cat([p1, p2], dim=1))

# --- 2. Residual Dense Block (RDB) ---
class RDB(nn.Module):
    def __init__(self, channels, growth_rate=32):
        super().__init__()
        # Dense layers: output of each is fed to all subsequent layers
        self.c1 = nn.Sequential(nn.Conv2d(channels, growth_rate, 3, 1, 1), nn.ReLU())
        self.c2 = nn.Sequential(nn.Conv2d(channels + growth_rate, growth_rate, 3, 1, 1), nn.ReLU())
        self.c3 = nn.Sequential(nn.Conv2d(channels + 2*growth_rate, channels, 3, 1, 1))

    def forward(self, x):
        out1 = self.c1(x)
        out2 = self.c2(torch.cat([x, out1], 1))
        out3 = self.c3(torch.cat([x, out1, out2], 1))
        return out3 + x  # Local Residual Connection

# --- 3. Enhanced Attention Module (EAM) ---
class EAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Channel Attention (Squeeze and Excitation)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.ca = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, 1),
            nn.Sigmoid()
        )
        # Spatial Attention (Focusing on 'where' the edges are)
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        # 1. Scale Channels
        x = x * self.ca(self.gap(x))
        # 2. Scale Pixels (Spatial)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_map = self.sa(torch.cat([avg_out, max_out], dim=1))
        return x * spatial_map

# --- 4. The Final Assembly Model ---
class InternshipProjectModel(nn.Module):
    def __init__(self, in_nc=3, out_nc=3, nf=64):
        super().__init__()
        self.msfe = MSFE(in_nc, nf)

        self.rdb1 = RDB(nf)
        self.rdb2 = RDB(nf)
        self.rdb3 = RDB(nf)

        # CNN Relay Bridge
        self.relay = nn.Conv2d(nf, nf, kernel_size=1)

        self.eam = EAM(nf)

        self.reconstruct = nn.Conv2d(nf, out_nc, kernel_size=3, padding=1)

    def forward(self, x):
        identity = x # Save input for Global Residual

        feat = self.msfe(x)

        # Backbone with Relay
        r1 = self.rdb1(feat)
        r2 = self.rdb2(r1)
        r3 = self.rdb3(r2)

        # Combine early (r1) and late (r3) features
        fused = self.relay(r1 + r3)

        # Filter and Output
        refined = self.eam(fused)
        return self.reconstruct(refined) + identity

4. How to Test It

In [10]:
# Create model
model = InternshipProjectModel()

# Create a dummy image (Batch size 1, 3 color channels, 64x64 pixels)
dummy_input = torch.randn(1, 3, 64, 64)

# Get output
output = model(dummy_input)

print(f"Input Shape: {dummy_input.shape}")
print(f"Output Shape: {output.shape}")
# Output should be [1, 3, 64, 64]

Input Shape: torch.Size([1, 3, 64, 64])
Output Shape: torch.Size([1, 3, 64, 64])
